# Semana 10.3 - Flujo optico y tracking

Notebook de taller con OpenCV + NumPy. Incluye:
1. Flujo optico disperso (Lucas-Kanade) con trayectorias y re-deteccion de puntos.
2. Flujo optico denso (Farneback) con visualizacion HSV en tiempo real.
3. Tracking de objeto por ROI manual con fallback robusto.
4. Estimacion de movimiento global de camara (pan/tilt/zoom).
5. Mascara de movimiento por magnitud de flujo y conteo de objetos moviles.
6. Analisis de rendimiento: FPS y comparacion de velocidad LK vs Farneback.

**Modo recomendado en notebook:** usar secuencia de imagenes (`INPUT_MODE = 'images'`) para ejecucion reproducible. Si quieres, puedes cambiar a `video` o `camera`.


In [ ]:
# Setup opcional para Google Colab (ejecutar solo en Colab)
import importlib.util
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])
    print('Dependencia verificada: opencv-python-headless')
else:
    print('Entorno local detectado: no se instala nada.')


In [ ]:
import glob
import os
import re
import time
import importlib.util
from collections import deque

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# ===============================
# Configuracion general
# ===============================
RUN_ENV = 'auto'  # 'auto' | 'colab' | 'local'
_IS_COLAB = importlib.util.find_spec('google.colab') is not None
USE_COLAB = _IS_COLAB if RUN_ENV == 'auto' else (RUN_ENV == 'colab')

INPUT_MODE = 'images'  # 'images' | 'video' | 'camera'
IMAGE_GLOB = '../media/*.jpg;../media/*.jpeg;../media/*.png;../media/*.bmp;../media/*.tif;../media/*.tiff'
VIDEO_SOURCE = '../media/video.mp4'  # Ruta a video cuando INPUT_MODE='video'
CAMERA_SOURCE = 0  # Indice de camara cuando INPUT_MODE='camera'
IMAGE_STEP_MS = 33  # Delay por frame en modo images/video cuando SHOW_WINDOWS=True
RESIZE_WIDTH = 960  # None para no redimensionar
SHOW_WINDOWS = not USE_COLAB  # Ventanas OpenCV solo en local
COLAB_DISPLAY_EVERY_N = 3  # Mostrar cada N frames para no saturar salida

ROI = None  # (x, y, w, h) para tracking sin GUI

# Lucas-Kanade (disperso)
MAX_CORNERS = 200
QUALITY_LEVEL = 0.3
MIN_DISTANCE = 7
BLOCK_SIZE = 7
LK_WIN_SIZE = (21, 21)
LK_MAX_LEVEL = 3
LK_CRITERIA = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
REDTECT_THRESHOLD = 60
TRACK_HISTORY = 20

# Farneback (denso)
FB_PYRSCALE = 0.5
FB_LEVELS = 3
FB_WINSIZE = 15
FB_ITERATIONS = 3
FB_POLY_N = 5
FB_POLY_SIGMA = 1.2
FB_FLAGS = 0

# Deteccion de movimiento
FLOW_MAG_THRESHOLD = 1.2
MIN_CONTOUR_AREA = 500

def natural_key(path):
    name = os.path.basename(path).lower()
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', name)]


def get_image_paths(image_glob):
    paths = []
    for pattern in [p.strip() for p in image_glob.split(';') if p.strip()]:
        paths.extend(glob.glob(pattern))
    return sorted(set(paths), key=natural_key)


def open_frame_source(input_mode, image_glob, video_source, camera_source):
    mode = input_mode.lower().strip()

    if mode == 'images':
        image_paths = get_image_paths(image_glob)
        if not image_paths:
            raise RuntimeError("No se encontraron imagenes. Ajusta IMAGE_GLOB, por ejemplo: '../media/*.jpg' o '../media/*.png'")
        print(f'Modo images: {len(image_paths)} imagenes cargadas.')

        def frame_iter():
            for path in image_paths:
                frame = cv2.imread(path)
                if frame is None:
                    print(f'Aviso: no se pudo leer {path}. Se omite.')
                    continue
                yield frame

        return frame_iter(), None

    if mode in ('video', 'camera'):
        source = video_source if mode == 'video' else camera_source
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            raise RuntimeError(f'No se pudo abrir la fuente {mode}: {source}')
        print(f'Modo {mode}: fuente {source}')

        def frame_iter():
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                yield frame

        return frame_iter(), cap

    raise ValueError("INPUT_MODE debe ser 'images', 'video' o 'camera'")


print('OpenCV:', cv2.__version__)
print('USE_COLAB:', USE_COLAB)
if USE_COLAB and INPUT_MODE == 'camera':
    print("Aviso: en Colab se recomienda INPUT_MODE='images' o 'video'.")


## Helpers: preprocesamiento, LK y visualizacion

In [ ]:
def resize_frame(frame, width=RESIZE_WIDTH):
    if width is None:
        return frame
    h, w = frame.shape[:2]
    if w == width:
        return frame
    scale = width / float(w)
    return cv2.resize(frame, (width, int(h * scale)), interpolation=cv2.INTER_AREA)


def detect_features(gray):
    return cv2.goodFeaturesToTrack(
        gray,
        mask=None,
        maxCorners=MAX_CORNERS,
        qualityLevel=QUALITY_LEVEL,
        minDistance=MIN_DISTANCE,
        blockSize=BLOCK_SIZE,
    )


def update_sparse_flow(prev_gray, gray, points, tracks):
    if points is None or len(points) == 0:
        points = detect_features(prev_gray)
        tracks.clear()
        return points, tracks, []

    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, gray, points, None,
        winSize=LK_WIN_SIZE,
        maxLevel=LK_MAX_LEVEL,
        criteria=LK_CRITERIA
    )

    if next_pts is None or status is None:
        points = detect_features(gray)
        tracks.clear()
        return points, tracks, []

    good_new = next_pts[status.flatten() == 1]
    good_old = points[status.flatten() == 1]

    vectors = []
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        x_new, y_new = new.ravel()
        x_old, y_old = old.ravel()
        vectors.append(((int(x_old), int(y_old)), (int(x_new), int(y_new))))

        if i >= len(tracks):
            tracks.append(deque(maxlen=TRACK_HISTORY))
        tracks[i].append((int(x_new), int(y_new)))

    tracks = tracks[: len(good_new)]
    points = good_new.reshape(-1, 1, 2)

    # Re-deteccion cuando se pierden demasiados puntos
    if len(points) < REDTECT_THRESHOLD:
        redetected = detect_features(gray)
        if redetected is not None:
            points = redetected
            tracks = [deque(maxlen=TRACK_HISTORY) for _ in range(len(points))]

    return points, tracks, vectors


def draw_sparse_overlay(frame, tracks, vectors):
    out = frame.copy()
    for tr in tracks:
        if len(tr) > 1:
            pts = np.array(tr, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(out, [pts], False, (0, 255, 255), 1)

    for p0, p1 in vectors:
        cv2.arrowedLine(out, p0, p1, (0, 255, 0), 1, tipLength=0.3)
        cv2.circle(out, p1, 2, (0, 0, 255), -1)

    cv2.putText(out, f'LK points: {len(vectors)}', (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    return out

## Helpers: Farneback denso, HSV, camara y mascara de movimiento

In [ ]:
def compute_dense_flow(prev_gray, gray):
    return cv2.calcOpticalFlowFarneback(
        prev_gray, gray, None,
        FB_PYRSCALE, FB_LEVELS, FB_WINSIZE,
        FB_ITERATIONS, FB_POLY_N, FB_POLY_SIGMA, FB_FLAGS
    )


def flow_to_hsv(flow):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
    hsv[..., 0] = (ang * 180 / np.pi / 2).astype(np.uint8)  # direccion -> hue
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)  # magnitud -> intensidad
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return bgr, mag, ang


def estimate_camera_motion(flow, mag):
    fx = flow[..., 0]
    fy = flow[..., 1]

    mean_fx = float(np.mean(fx))
    mean_fy = float(np.mean(fy))
    std_mag = float(np.std(mag))
    mean_mag = float(np.mean(mag))

    h, w = fx.shape
    yy, xx = np.mgrid[0:h, 0:w]
    cx, cy = w / 2.0, h / 2.0
    radial = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2) + 1e-6
    ux = (xx - cx) / radial
    uy = (yy - cy) / radial
    radial_component = fx * ux + fy * uy
    zoom_indicator = float(np.mean(radial_component))

    pan = 'right' if mean_fx > 0.2 else 'left' if mean_fx < -0.2 else 'stable'
    tilt = 'down' if mean_fy > 0.2 else 'up' if mean_fy < -0.2 else 'stable'
    zoom = 'in' if zoom_indicator > 0.1 else 'out' if zoom_indicator < -0.1 else 'stable'

    return {
        'pan': pan,
        'tilt': tilt,
        'zoom': zoom,
        'vx_px': mean_fx,
        'vy_px': mean_fy,
        'zoom_px': zoom_indicator,
        'mean_mag': mean_mag,
        'std_mag': std_mag,
    }


def motion_mask_from_magnitude(mag):
    mask = (mag > FLOW_MAG_THRESHOLD).astype(np.uint8) * 255
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        area = cv2.contourArea(c)
        if area >= MIN_CONTOUR_AREA:
            boxes.append(cv2.boundingRect(c))

    return mask, boxes

## Helpers: tracking por ROI con fallback

Si no hay trackers disponibles en tu build de OpenCV, se usa fallback por template matching.

In [ ]:
def build_tracker():
    factories = []

    # APIs modernas y legacy posibles
    if hasattr(cv2, 'legacy'):
        legacy = cv2.legacy
        for name in ['TrackerCSRT_create', 'TrackerKCF_create', 'TrackerMOSSE_create']:
            if hasattr(legacy, name):
                factories.append(getattr(legacy, name))

    for name in ['TrackerCSRT_create', 'TrackerKCF_create', 'TrackerMIL_create']:
        if hasattr(cv2, name):
            factories.append(getattr(cv2, name))

    for fn in factories:
        try:
            return fn(), fn.__name__.replace('_create', '')
        except Exception:
            pass

    return None, 'TemplateMatchFallback'


def show_roi_helper(frame):
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title('Frame inicial para definir ROI=(x, y, w, h)')
    plt.axis('off')
    plt.show()
    print('Define ROI manualmente en la celda de configuracion. Ejemplo: ROI = (120, 80, 160, 140)')


def init_tracking(frame, roi_config=None):
    if roi_config is None:
        if SHOW_WINDOWS and not USE_COLAB:
            roi = cv2.selectROI('Select ROI then ENTER', frame, fromCenter=False, showCrosshair=True)
            cv2.destroyWindow('Select ROI then ENTER')
        else:
            show_roi_helper(frame)
            return None
    else:
        roi = roi_config

    x, y, w, h = [int(v) for v in roi]
    if w <= 0 or h <= 0:
        return None

    tracker, tracker_name = build_tracker()
    template = frame[y:y+h, x:x+w].copy()

    state = {
        'bbox': (x, y, w, h),
        'ok': True,
        'lost_count': 0,
        'tracker': tracker,
        'tracker_name': tracker_name,
        'template': template,
    }

    if tracker is not None:
        state['ok'] = tracker.init(frame, (x, y, w, h))

    return state


def update_tracking(frame, state):
    if state is None:
        return None, 'not_initialized'

    h_frame, w_frame = frame.shape[:2]
    tracking_ok = False

    if state['tracker'] is not None:
        ok, bbox = state['tracker'].update(frame)
        if ok:
            x, y, w, h = [int(v) for v in bbox]
            tracking_ok = (w > 0 and h > 0)
            if tracking_ok:
                state['bbox'] = (x, y, w, h)
    else:
        x, y, w, h = state['bbox']
        roi = frame[max(0, y-30):min(h_frame, y+h+30), max(0, x-30):min(w_frame, x+w+30)]
        if roi.size > 0 and state['template'].size > 0 and roi.shape[0] >= h and roi.shape[1] >= w:
            res = cv2.matchTemplate(roi, state['template'], cv2.TM_CCOEFF_NORMED)
            _, max_val, _, max_loc = cv2.minMaxLoc(res)
            if max_val > 0.45:
                x0 = max(0, x - 30) + max_loc[0]
                y0 = max(0, y - 30) + max_loc[1]
                state['bbox'] = (x0, y0, w, h)
                tracking_ok = True

    x, y, w, h = state['bbox']
    x = max(0, min(x, w_frame - 1))
    y = max(0, min(y, h_frame - 1))
    w = max(1, min(w, w_frame - x))
    h = max(1, min(h, h_frame - y))
    state['bbox'] = (x, y, w, h)

    # Heuristica de perdida / oclusion parcial
    crop = frame[y:y+h, x:x+w]
    occlusion_suspected = False
    if crop.size > 0:
        gray_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        var_lap = cv2.Laplacian(gray_crop, cv2.CV_64F).var()
        mean_intensity = float(np.mean(gray_crop))
        if var_lap < 8.0 or mean_intensity < 20 or mean_intensity > 235:
            occlusion_suspected = True

    if not tracking_ok:
        state['lost_count'] += 1
    else:
        state['lost_count'] = 0

    if state['lost_count'] > 10:
        return state, 'lost'
    if occlusion_suspected:
        return state, 'partial_occlusion'
    return state, 'ok' if tracking_ok else 'recovering'


## Ejecucion principal

Atajos (solo en ventanas OpenCV locales):
- `q`: salir
- `r`: re-seleccionar ROI

En Colab no se usan teclas: el bucle termina al finalizar frames de `images`/`video`.
Para tracking en Colab define `ROI = (x, y, w, h)` en la configuracion.


In [ ]:
frames, cap = open_frame_source(INPUT_MODE, IMAGE_GLOB, VIDEO_SOURCE, CAMERA_SOURCE)

first_frame = next(frames, None)
if first_frame is None:
    if cap is not None:
        cap.release()
    raise RuntimeError('No hay frames disponibles en la fuente seleccionada.')

frame = resize_frame(first_frame)
prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
points = detect_features(prev_gray)
tracks = [deque(maxlen=TRACK_HISTORY) for _ in range(0 if points is None else len(points))]
tracker_state = init_tracking(frame, ROI)
tracking_enabled = tracker_state is not None
if not tracking_enabled:
    print('Tracking ROI deshabilitado: define ROI=(x, y, w, h) para activarlo.')

time_prev = time.perf_counter()
fps_hist = deque(maxlen=40)
lk_times = deque(maxlen=200)
fb_times = deque(maxlen=200)

print('Iniciando procesamiento...')
if USE_COLAB:
    print('Modo Colab: visualizacion periodica con matplotlib.')

for idx, frame in enumerate(frames, start=1):
    frame = resize_frame(frame)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    t0 = time.perf_counter()
    points, tracks, vectors = update_sparse_flow(prev_gray, gray, points, tracks)
    t1 = time.perf_counter()
    lk_ms = (t1 - t0) * 1000.0
    lk_times.append(lk_ms)

    t2 = time.perf_counter()
    flow = compute_dense_flow(prev_gray, gray)
    dense_bgr, mag, ang = flow_to_hsv(flow)
    t3 = time.perf_counter()
    fb_ms = (t3 - t2) * 1000.0
    fb_times.append(fb_ms)

    if tracking_enabled:
        tracker_state, track_status = update_tracking(frame, tracker_state)
    else:
        track_status = 'disabled'

    cam = estimate_camera_motion(flow, mag)
    motion_mask, motion_boxes = motion_mask_from_magnitude(mag)

    now = time.perf_counter()
    fps = 1.0 / max(now - time_prev, 1e-6)
    time_prev = now
    fps_hist.append(fps)
    fps_avg = float(np.mean(fps_hist)) if fps_hist else fps

    sparse_vis = draw_sparse_overlay(frame, tracks, vectors)
    motion_vis = frame.copy()
    for (x, y, w, h) in motion_boxes:
        cv2.rectangle(motion_vis, (x, y), (x + w, y + h), (255, 180, 0), 2)

    cv2.putText(motion_vis, f'Moving objects: {len(motion_boxes)}', (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 180, 0), 2)

    if tracking_enabled and tracker_state is not None:
        x, y, w, h = tracker_state['bbox']
        color = (0, 255, 0) if track_status == 'ok' else (0, 200, 255) if track_status == 'partial_occlusion' else (0, 0, 255)
        cv2.rectangle(motion_vis, (x, y), (x + w, y + h), color, 2)
        cv2.putText(motion_vis, f"Tracker: {tracker_state['tracker_name']} | {track_status}", (10, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    cv2.putText(sparse_vis, f'FPS: {fps_avg:.1f}', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
    cv2.putText(sparse_vis, f'LK: {lk_ms:.2f} ms | FB: {fb_ms:.2f} ms', (10, 76), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    cv2.putText(sparse_vis, f"Cam pan:{cam['pan']} tilt:{cam['tilt']} zoom:{cam['zoom']}", (10, 102), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)
    cv2.putText(sparse_vis, f"v=({cam['vx_px']:.2f}, {cam['vy_px']:.2f}) px/frame  z={cam['zoom_px']:.2f}", (10, 126), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

    if SHOW_WINDOWS:
        cv2.imshow('1) Sparse LK', sparse_vis)
        cv2.imshow('2) Dense Farneback HSV', dense_bgr)
        cv2.imshow('3) Tracking + Motion', motion_vis)
        cv2.imshow('4) Motion Mask', motion_mask)

        key = cv2.waitKey(1 if INPUT_MODE == 'camera' else IMAGE_STEP_MS) & 0xFF
        if key == ord('q'):
            break
        if key == ord('r') and not USE_COLAB:
            tracker_state = init_tracking(frame, None)
            tracking_enabled = tracker_state is not None
    elif USE_COLAB and (idx % COLAB_DISPLAY_EVERY_N == 0):
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes[0, 0].imshow(cv2.cvtColor(sparse_vis, cv2.COLOR_BGR2RGB))
        axes[0, 0].set_title('1) Sparse LK')
        axes[0, 1].imshow(cv2.cvtColor(dense_bgr, cv2.COLOR_BGR2RGB))
        axes[0, 1].set_title('2) Dense Farneback HSV')
        axes[1, 0].imshow(cv2.cvtColor(motion_vis, cv2.COLOR_BGR2RGB))
        axes[1, 0].set_title('3) Tracking + Motion')
        axes[1, 1].imshow(motion_mask, cmap='gray')
        axes[1, 1].set_title('4) Motion Mask')
        for ax in axes.ravel():
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    prev_gray = gray

if cap is not None:
    cap.release()
if SHOW_WINDOWS:
    cv2.destroyAllWindows()

if INPUT_MODE == 'images':
    print('Fin de secuencia de imagenes.')

lk_mean = float(np.mean(lk_times)) if lk_times else float('nan')
fb_mean = float(np.mean(fb_times)) if fb_times else float('nan')
print(f'Promedio LK: {lk_mean:.3f} ms/frame')
print(f'Promedio FB: {fb_mean:.3f} ms/frame')
if np.isfinite(lk_mean) and np.isfinite(fb_mean) and lk_mean > 0:
    print(f'FB/LK speed ratio: {fb_mean / lk_mean:.2f}x')


## Comparacion de rendimiento (grafica)

In [ ]:
if len(lk_times) > 0 and len(fb_times) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot(lk_times, label='Lucas-Kanade ms/frame', linewidth=1.2)
    plt.plot(fb_times, label='Farneback ms/frame', linewidth=1.2)
    plt.xlabel('Frame')
    plt.ylabel('Tiempo (ms)')
    plt.title('Comparacion de costo computacional')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No hay muestras suficientes para graficar.')